# Pipe Profile Step Visualizer

This notebook expands `calculate_pipe_profile()` into visible steps.

In [10]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path


import numpy as np
import open3d as o3d
from pyvista import Plotter

from EndEffectorPoseOptimizer import EndEffectorPoseOptimizer
from CylinderFitting import fit_cylinder
from JupyterVisualizer import (
    add_box,
    add_cylinder,
    add_point_clusters,
    add_points,
    add_sphere,
    visualize_pointclouds_simply,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Inputs

In [11]:
SCAN_PATH = "./data/PIPE NO.3_fill.ply"
SCAN_SCALE = 1
TARGET_POINT_IDX = 303135

sampling_size_for_calculating_normal = 0.01
radius_offset_for_sampling_points_in_sphere = 0.003
cylinder_roi_radius = 0.005
cylinder_height_range = (-0.1, 0.3)
cluster_distance = 0.005

## 1. Load scan and target point

In [12]:
optimizer = EndEffectorPoseOptimizer(debug_mode=True)
optimizer.load_scan_data(SCAN_PATH, scale=SCAN_SCALE)

if not optimizer._scan_data.has_normals():
    optimizer._scan_data.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=30)
    )

points = np.asarray(optimizer._scan_data.points)
target_point = np.asarray(optimizer._scan_data.points[TARGET_POINT_IDX])

print("point_count:", len(points))
print("bounds_min:", points.min(axis=0))
print("bounds_max:", points.max(axis=0))
print("target_point:", target_point)

point_count: 1622355
bounds_min: [-0.44829999 -5.255      -0.075     ]
bounds_max: [0.15       0.03749988 0.5175    ]
target_point: [ 0.03510616 -2.34402818 -0.01382356]


In [13]:
optimizer._scan_data

PointCloud with 1622355 points.

In [14]:
pl = Plotter()
visualize_pointclouds_simply(optimizer._scan_data, plotter=pl, point_size=2)
add_sphere(pl, target_point, radius=0.01, color="black")
pl.show()

Widget(value='<iframe src="http://localhost:50978/index.html?ui=P_0x2c227c2fdc0_3&reconnect=auto" class="pyvis…

## 2. Sample box and median normal

In [15]:
gap = np.full(3, sampling_size_for_calculating_normal, dtype=np.float64)
min_bound = target_point - gap
max_bound = target_point + gap
box = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)  # type: ignore

indices = box.get_point_indices_within_bounding_box(optimizer._scan_data.points)
selected_points = optimizer._scan_data.select_by_index(indices)
selected_normals = np.asarray(selected_points.normals)
normal_m = np.median(selected_normals, axis=0)

print("selected_count:", len(selected_points.points))
print("normal_m:", normal_m)
print("normal_norm:", np.linalg.norm(normal_m))

selected_count: 414
normal_m: [ 0.95693426 -0.00105053 -0.29029018]
normal_norm: 0.999996331129928


In [16]:
pl = Plotter()
visualize_pointclouds_simply(optimizer._scan_data, plotter=pl, point_size=2)
add_box(pl, min_bound, max_bound, color="red", opacity=0.25)
add_points(pl, selected_points, point_size=8)
add_sphere(pl, target_point, radius=0.01, color="black")
pl.add_arrows(target_point.reshape(1, 3), normal_m.reshape(1, 3), 0.08, color="red")
pl.add_arrows(target_point.reshape(1, 3), (-normal_m).reshape(1, 3), 0.08, color="blue")
pl.show()

Widget(value='<iframe src="http://localhost:50978/index.html?ui=P_0x2c227c2ed70_4&reconnect=auto" class="pyvis…

## 3. Thin cylinder ROI

In [17]:
normal_axis = normal_m * -1
points_in_cylinder = optimizer._EndEffectorPoseOptimizer__extract_points_in_cylinder(
    points,
    target_point,
    normal_axis,
    cylinder_roi_radius,
    cylinder_height_range,
)

print("points_in_cylinder:", len(points_in_cylinder))

points_in_cylinder: 174


In [18]:
pl = Plotter()
visualize_pointclouds_simply(optimizer._scan_data, plotter=pl, point_size=2)
cylinder_center = target_point + normal_axis * np.mean(cylinder_height_range)
cylinder_height = cylinder_height_range[1] - cylinder_height_range[0]
add_cylinder(pl, cylinder_center, normal_axis, cylinder_roi_radius, cylinder_height, color="orange", opacity=0.25)
add_point_clusters(pl, [points_in_cylinder], point_size=12, colors=["orange"])
add_sphere(pl, target_point, radius=0.01, color="black")
pl.add_arrows(target_point.reshape(1, 3), normal_axis.reshape(1, 3), 0.12, color="blue")
pl.show()

Widget(value='<iframe src="http://localhost:50978/index.html?ui=P_0x2c230d9de10_5&reconnect=auto" class="pyvis…

## 4. Projection clusters

In [19]:
clusters = optimizer._EndEffectorPoseOptimizer__cluster_points_along_line(
    points_in_cylinder,
    target_point,
    normal_axis,
    cluster_distance,
)

print("cluster_count:", len(clusters))
print("cluster_sizes:", [len(cluster) for cluster in clusters])

cluster_count: 2
cluster_sizes: [97, 77]


In [ ]:
pl = Plotter()
visualize_pointclouds_simply(optimizer._scan_data, plotter=pl, point_size=2)
add_point_clusters(pl, clusters, point_size=14,  colors=["orange"])
add_sphere(pl, target_point, radius=0.01, color="black")
pl.add_arrows(target_point.reshape(1, 3), normal_axis.reshape(1, 3), 0.12, color="blue")
pl.show()

Widget(value='<iframe src="http://localhost:50978/index.html?ui=P_0x2c29705bf70_8&reconnect=auto" class="pyvis…

## 5. Estimated opposite point, center, and radius

In [ ]:
if len(clusters) < 2:
    raise RuntimeError(f"Only {len(clusters)} cluster(s). The original clusters[1] assumption fails here.")

estimated_opposite_point = np.asarray(clusters[1][-1])
estimated_center = (target_point + estimated_opposite_point) / 2
estimated_radius = float(np.linalg.norm(estimated_opposite_point - estimated_center))

print("estimated_opposite_point:", estimated_opposite_point)
print("estimated_center:", estimated_center)
print("estimated_radius:", estimated_radius)

In [ ]:
pl = Plotter()
visualize_pointclouds_simply(optimizer._scan_data, plotter=pl, point_size=2)
add_point_clusters(pl, clusters, point_size=12)
add_sphere(pl, target_point, radius=0.01, color="black")
add_sphere(pl, estimated_opposite_point, radius=0.01, color="white")
add_sphere(pl, estimated_center, radius=estimated_radius, color="red", opacity=0.2)
pl.show()

## 6. Sphere ROI and cylinder fit

In [ ]:
points_in_sphere = optimizer._EndEffectorPoseOptimizer__extract_points_in_sphere(
    points,
    estimated_center,
    estimated_radius + radius_offset_for_sampling_points_in_sphere,
)

direction, center, radius, _ = fit_cylinder(points_in_sphere)

print("points_in_sphere:", len(points_in_sphere))
print("fitted_center:", center)
print("fitted_direction:", direction)
print("fitted_radius:", radius)

In [ ]:
pl = Plotter()
visualize_pointclouds_simply(optimizer._scan_data, plotter=pl, point_size=2)
add_point_clusters(pl, [points_in_sphere], point_size=8, colors=["orange"])
add_sphere(pl, target_point, radius=0.01, color="black")
pl.add_arrows(center.reshape(1, 3), direction.reshape(1, 3), 0.12, color="red")
add_cylinder(pl, center, direction, radius + radius_offset_for_sampling_points_in_sphere, 0.08, "blue", 0.25)
pl.show()